Library

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision, BaseOptions
import tensorflow as tf
from tensorflow.keras.applications.mobilenet import preprocess_input
import os
import time
import psutil
import pandas as pd
import torch
import torch.nn as nn
import timm
from torchvision import transforms, models
from pytorchvideo.models.hub import x3d_s

Preprocessing

Hand Detector Setup

In [2]:
MODEL_PATH = "hand_landmarker.task"

base_options = BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(options)

Load Video Frames Function

In [3]:
def load_frames_from_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames

Making Box Function

In [4]:
def get_static_hand_bbox(frames, padding=0.2):
    global_min_x, global_min_y = 1.0, 1.0
    global_max_x, global_max_y = 0.0, 0.0

    found = False
    h, w = frames[0].shape[:2]

    for frame in frames:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        result = detector.detect(mp_image)

        if result.hand_landmarks:
            found = True
            for hand in result.hand_landmarks:
                for lm in hand:
                    global_min_x = min(global_min_x, lm.x)
                    global_min_y = min(global_min_y, lm.y)
                    global_max_x = max(global_max_x, lm.x)
                    global_max_y = max(global_max_y, lm.y)

    if not found:
        # fallback: return full frame
        return 0, 0, w, h

    # convert to pixels
    box_w = global_max_x - global_min_x
    box_h = global_max_y - global_min_y

    x1 = max(0, int((global_min_x - padding * box_w) * w))
    y1 = max(0, int((global_min_y - padding * box_h) * h))
    x2 = min(w, int((global_max_x + padding * box_w) * w))
    y2 = min(h, int((global_max_y + padding * box_h) * h))

    return x1, y1, x2, y2

Crop Frames Function

In [ ]:
def crop_frames_with_bbox(frames, bbox):
    x1, y1, x2, y2 = bbox

    cropped_frames = []
    for f in frames:
        crop = f[y1:y2, x1:x2]
        cropped_frames.append(crop)

    return cropped_frames

Dynamic Image Test Setup

Dynamic Image Function

In [ ]:
def get_dynamic_image(frames, normalized=True):
    num_channels = frames[0].shape[2]
    channel_frames = _get_channel_frames(frames, num_channels)
    channel_dynamic_images = [_compute_dynamic_image(channel) for channel in channel_frames]

    dynamic_image = cv2.merge(tuple(channel_dynamic_images))
    if normalized:
        dynamic_image = cv2.normalize(dynamic_image, None, 0, 255, norm_type=cv2.NORM_MINMAX)
        dynamic_image = dynamic_image.astype('uint8')

    return dynamic_image


def _get_channel_frames(iter_frames, num_channels):
    frames = [[] for channel in range(num_channels)]

    for frame in iter_frames:
        for channel_frames, channel in zip(frames, cv2.split(frame)):
            channel_frames.append(channel.reshape((*channel.shape[0:2], 1)))
    for i in range(len(frames)):
        frames[i] = np.array(frames[i])
    return frames

def _compute_dynamic_image(frames):
    """ Adapted from https://github.com/hbilen/dynamic-image-nets """
    num_frames, h, w, depth = frames.shape

    # Compute the coefficients for the frames.
    coefficients = np.zeros(num_frames)
    for n in range(num_frames):
        cumulative_indices = np.array(range(n, num_frames)) + 1
        coefficients[n] = np.sum(((2*cumulative_indices) - num_frames) / cumulative_indices)

    # Multiply by the frames by the coefficients and sum the result.
    x1 = np.expand_dims(frames, axis=0)
    x2 = np.reshape(coefficients, (num_frames, 1, 1, 1))
    result = x1 * x2
    return np.sum(result[0], axis=0).squeeze()

#Reading Frames from Video
def load_frames_opencv(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames

Convert from video to dynamic image

In [ ]:
def video_to_dynamic_image(video_path, output_size=(224,224)):

    frames = load_frames_opencv(video_path)
    if len(frames) == 0:
        raise ValueError("No frames found in video")

    #Preprocessing
    bbox = get_static_hand_bbox(frames)
    cropped_frames = crop_frames_with_bbox(frames, bbox)

    # Create dynamic image 
    dyn_img = get_dynamic_image(cropped_frames)

    # Resize 
    dyn_img = cv2.resize(dyn_img, output_size)

    return dyn_img

Test Time and Memory Usage

MobileNet

In [ ]:


process = psutil.Process(os.getpid())


# I use sign_455783 as testing
video_folder = "test\CAR"
model_path = "final_mobilenet_model_final, 63.38.keras"

TRUE_LABEL = "CAR"

class_names = ['BUY','CAR','Eat','FRIEND','GO','GO-OUT',
               'HOUSE','Have','KISS-FIST','LOOK','Must',
               'NOT','NOW','PEOPLE','TEND']

model = tf.keras.models.load_model(model_path)

results = []

for video in os.listdir(video_folder):

    video_path = os.path.join(video_folder, video)

    #Start Time
    start_time = time.time()
    start_mem = process.memory_info().rss / (1024**2)

    #Change vid -> Dynamic Img
    dyn_img = video_to_dynamic_image(video_path)

    dyn_img = dyn_img.astype("float32")
    dyn_img = preprocess_input(dyn_img)
    dyn_img = np.expand_dims(dyn_img, axis=0)

    pred = model.predict(dyn_img, verbose=0)
    pred_class = np.argmax(pred)
    pred_label = class_names[pred_class]

    #Stop Time
    end_time = time.time()
    end_mem = process.memory_info().rss / (1024**2)

    #Get data for time and memory consumption
    results.append({
        "video": video,
        "true_label": TRUE_LABEL,
        "pred_label": pred_label,
        "correct": int(pred_label == TRUE_LABEL),
        "time_sec": end_time - start_time,
        "memory_MB": end_mem - start_mem
    })

#Get Result
df = pd.DataFrame(results)

df.to_csv("single_class_test.csv", index=False)

print(df)

print("\nAccuracy (this class only):", df["correct"].mean())
print("Avg Time:", df["time_sec"].mean())
print("Avg Memory:", df["memory_MB"].mean())



             video true_label pred_label  correct  time_sec  memory_MB
0  sign_379329.mp4        CAR        NOT        0  1.090999  22.179688
1  sign_387006.mp4        CAR        NOW        0  0.671619  24.437500
2  sign_388681.mp4        CAR        CAR        1  0.666567   2.226562

Accuracy (this class only): 0.3333333333333333
Avg Time: 0.8097285429636637
Avg Memory: 16.28125


EffecientNet

In [ ]:
process = psutil.Process(os.getpid())

# CONFIG
video_folder = "test\CAR"
model_path = "efficientnet_0.5714.pth"

TRUE_LABEL = "CAR"

class_names = ['BUY','CAR','Eat','FRIEND','GO','GO-OUT',
               'HOUSE','Have','KISS-FIST','LOOK','Must',
               'NOW','TEND']

device = "cuda" if torch.cuda.is_available() else "cpu"

# LOAD MODEL 
model = timm.create_model("efficientnet_b0", pretrained=False)

in_features = model.get_classifier().in_features

model.classifier = nn.Sequential(
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.6),
    nn.Linear(in_features, len(class_names))
)

model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

print("Model loaded!")

# DYNAMIC IMAGE
def video_to_dynamic_image(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (224, 224))
        frames.append(frame.astype(np.float32))

    cap.release()

    if len(frames) == 0:
        return np.zeros((224, 224, 3), dtype=np.float32)

    weights = np.arange(1, len(frames) + 1)
    weights = weights / weights.sum()

    dyn_img = np.zeros_like(frames[0])
    for i in range(len(frames)):
        dyn_img += weights[i] * frames[i]

    return dyn_img

# SIMPLE PREPROCESS 
def to_tensor(img):
    img = img.astype(np.float32)        # keep raw values
    img = np.transpose(img, (2, 0, 1))  # HWC → CHW
    img = np.expand_dims(img, axis=0)   # add batch
    return torch.tensor(img).to(device)

# INFERENCE LOOP
results = []
input_tensor = to_tensor(dyn_img)
for video in os.listdir(video_folder):
    if not video.lower().endswith(('.mp4', '.avi', '.mov')):
        continue
    video_path = os.path.join(video_folder, video)
    start_time = time.time()
    start_mem = process.memory_info().rss / (1024**2)
    dyn_img = video_to_dynamic_image(video_path)

  

    # Predict
    with torch.no_grad():
        outputs = model(input_tensor)
        pred_class = torch.argmax(outputs, dim=1).item()

    pred_label = class_names[pred_class]

    end_time = time.time()
    end_mem = process.memory_info().rss / (1024**2)

    results.append({
        "video": video,
        "true_label": TRUE_LABEL,
        "pred_label": pred_label,
        "correct": int(pred_label == TRUE_LABEL),
        "time_sec": end_time - start_time,
        "memory_MB": end_mem - start_mem
    })

# RESULTS
df = pd.DataFrame(results)

df.to_csv("single_class_test.csv", index=False)

print("\nResults:")
print(df)

print("\nAccuracy (this class only):", df["correct"].mean())
print("Avg Time:", df["time_sec"].mean())
print("Avg Memory:", df["memory_MB"].mean())

Model loaded!
Processing: sign_379329.mp4
Processing: sign_387006.mp4
Processing: sign_388681.mp4

Results:
             video true_label pred_label  correct  time_sec  memory_MB
0  sign_379329.mp4        CAR       Must        0  0.066715  11.820312
1  sign_387006.mp4        CAR       Must        0  0.067714  11.226562
2  sign_388681.mp4        CAR       Must        0  0.073742   0.207031

Accuracy (this class only): 0.0
Avg Time: 0.06939053535461426
Avg Memory: 7.751302083333333


ResNet

In [ ]:

process = psutil.Process(os.getpid())

# CONFIG
video_folder = "test\CAR"
model_path = "resnet18_0.5481.pth"

TRUE_LABEL = "HOUSE"

class_names = ['BUY','CAR','Eat','FRIEND','GO','GO-OUT',
               'HOUSE','Have','KISS-FIST','LOOK','Must',
               'NOW','TEND']

device = "cuda" if torch.cuda.is_available() else "cpu"

# LOAD MODEL 
model = timm.create_model("resnet18", pretrained=False)

in_features = model.get_classifier().in_features

model.fc = nn.Sequential(
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.6),
    nn.Linear(in_features, len(class_names))
)

model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# SIMPLE PREPROCESS (FORMAT ONLY)
def to_tensor(img):
    img = img.astype(np.float32)
    img = np.transpose(img, (2, 0, 1))  # HWC → CHW
    img = np.expand_dims(img, axis=0)   # add batch
    return torch.tensor(img).to(device)

# INFERENCE LOOP
results = []

for video in os.listdir(video_folder):

    if not video.lower().endswith(('.mp4', '.avi', '.mov')):
        continue

    video_path = os.path.join(video_folder, video)
    start_time = time.time()
    start_mem = process.memory_info().rss / (1024**2)

    # Video to Dynamic Image
    dyn_img = video_to_dynamic_image(video_path)
    input_tensor = to_tensor(dyn_img)

    # Predict
    with torch.no_grad():
        outputs = model(input_tensor)
        pred_class = torch.argmax(outputs, dim=1).item()

    pred_label = class_names[pred_class]

    end_time = time.time()
    end_mem = process.memory_info().rss / (1024**2)

    results.append({
        "video": video,
        "true_label": TRUE_LABEL,
        "pred_label": pred_label,
        "correct": int(pred_label == TRUE_LABEL),
        "time_sec": end_time - start_time,
        "memory_MB": end_mem - start_mem
    })

# RESULTS
df = pd.DataFrame(results)

df.to_csv("single_class_test.csv", index=False)

print("\nResults:")
print(df)

print("\nAccuracy (this class only):", df["correct"].mean())
print("Avg Time:", df["time_sec"].mean())
print("Avg Memory:", df["memory_MB"].mean())

Model loaded!
Processing: sign_379329.mp4
Processing: sign_387006.mp4
Processing: sign_388681.mp4

Results:
             video true_label pred_label  correct  time_sec  memory_MB
0  sign_379329.mp4      HOUSE         GO        0  0.047346  25.863281
1  sign_387006.mp4      HOUSE         GO        0  0.042627  -1.039062
2  sign_388681.mp4      HOUSE         GO        0  0.053771  -1.355469

Accuracy (this class only): 0.0
Avg Time: 0.04791482289632162
Avg Memory: 7.822916666666667


Video Processing Test Setup

Video Functions

In [ ]:
# FRAME SAMPLING
def sample_frames(frames, num_frames):

    total = len(frames)

    if total < num_frames:
        frames += [frames[-1]] * (num_frames - total)
        return frames

    indices = np.linspace(0, total - 1, num_frames).astype(int)
    return [frames[i] for i in indices]

# VIDEO to TENSOR
def video_to_tensor(video_path):

    frames = load_frames(video_path)

    if len(frames) == 0:
        raise ValueError("No frames in video")

    x1, y1, x2, y2 = get_static_hand_bbox(frames)

    cropped_frames = [f[y1:y2, x1:x2] for f in frames]
    sampled_frames = sample_frames(cropped_frames, NUM_FRAMES)

    processed = [test_transform(f) for f in sampled_frames]

    return torch.stack(processed)  # (T,C,H,W)

MobileNetV2

In [ ]:

# CONFIG
NUM_FRAMES = 16
IMG_SIZE = 224

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

video_folder = "test\CAR"  # CHANGE THIS
model_path = "mobilenetv2_video_model.pth"

TRUE_LABEL = "MUST"

class_names = ['BUY','CAR','EAT','FRIEND','GO','GO-OUT',
               'HOUSE','HAVE','KISS-FIST','LOOK','MUST',
               'NOW','TEND']

# MEDIAPIPE HAND LANDMARKER
MODEL_PATH = "hand_landmarker.task"

base_options = BaseOptions(model_asset_path=MODEL_PATH)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(options)

# TRANSFORM 
test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# MODEL
class MobileNetVideoClassifier(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        backbone = models.mobilenet_v2(pretrained=False)
        self.feature_extractor = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(1280, num_classes)
        )

    def forward(self, x):

        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)

        features = self.feature_extractor(x)
        features = self.pool(features)

        features = features.view(B*T, 1280)
        features = features.view(B, T, 1280)

        features = features.mean(dim=1)

        return self.classifier(features)

# LOAD MODEL
model = MobileNetVideoClassifier(len(class_names)).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# TESTING
process = psutil.Process(os.getpid())
results = []

# Handle both file & folder
if os.path.isfile(video_folder):
    video_list = [video_folder]
else:
    video_list = [
        os.path.join(video_folder, v)
        for v in os.listdir(video_folder)
        if v.lower().endswith((".mp4", ".avi", ".mov"))
    ]

for video_path in video_list:

    video_name = os.path.basename(video_path)

    start_time = time.time()
    start_mem = process.memory_info().rss / (1024**2)

    video_tensor = video_to_tensor(video_path)
    video_tensor = video_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(video_tensor)
        pred_class = outputs.argmax(1).item()
        pred_label = class_names[pred_class]

    end_time = time.time()
    end_mem = process.memory_info().rss / (1024**2)

    results.append({
        "video": video_name,
        "true_label": TRUE_LABEL,
        "pred_label": pred_label,
        "correct": int(pred_label == TRUE_LABEL),
        "time_sec": end_time - start_time,
        "memory_MB": end_mem - start_mem
    })

# RESULT
df = pd.DataFrame(results)

df.to_csv("video_classification_results.csv", index=False)

print(df)

print("\nAccuracy:", df["correct"].mean())
print("Avg Time:", df["time_sec"].mean())
print("Avg Memory:", df["memory_MB"].mean())

Using device: cpu


c:\Users\Kazumi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Kazumi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


             video true_label pred_label  correct  time_sec   memory_MB
0  sign_379329.mp4       MUST        CAR        0  0.468108  204.933594
1  sign_387006.mp4       MUST        CAR        0  0.678317    5.980469
2  sign_388681.mp4       MUST        CAR        0  0.640897  -19.781250

Accuracy: 0.0
Avg Time: 0.5957740147908529
Avg Memory: 63.7109375


EffecientNet

In [ ]:
# CONFIG
NUM_FRAMES = 12   # MUST match training
IMG_SIZE = 224

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

video_folder = "test\CAR"
model_path = "efficientnet_13_classess_final.pth"

TRUE_LABEL = "House"

class_names = ['BUY','CAR','Eat','FRIEND','GO','GO-OUT',
               'HOUSE','Have','KISS-FIST','LOOK','Must',
               'NOW','TEND']

# MEDIAPIPE HAND LANDMARKER
MODEL_PATH = "hand_landmarker.task"

base_options = BaseOptions(model_asset_path=MODEL_PATH)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(options)

# Augmentation
test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# MODEL (EfficientNet + LSTM)
class EfficientNetLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=False,
            num_classes=0,
            global_pool="avg"
        )

        self.feature_dim = self.backbone.num_features

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=256,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.view(B*T, C, H, W)
        feats = self.backbone(x)

        feats = feats.view(B, T, -1)

        lstm_out, _ = self.lstm(feats)
        lstm_out = lstm_out[:, -1, :]

        return self.classifier(lstm_out)

# LOAD MODEL
model = EfficientNetLSTM(len(class_names)).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

# TESTING
process = psutil.Process(os.getpid())
results = []

if os.path.isfile(video_folder):
    video_list = [video_folder]
else:
    video_list = [
        os.path.join(video_folder, v)
        for v in os.listdir(video_folder)
        if v.lower().endswith((".mp4", ".avi", ".mov"))
    ]

for video_path in video_list:

    video_name = os.path.basename(video_path)

    start_time = time.time()
    start_mem = process.memory_info().rss / (1024**2)

    video_tensor = video_to_tensor(video_path)
    video_tensor = video_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(video_tensor)
        pred_class = outputs.argmax(1).item()
        pred_label = class_names[pred_class]

    end_time = time.time()
    end_mem = process.memory_info().rss / (1024**2)

    results.append({
        "video": video_name,
        "true_label": TRUE_LABEL,
        "pred_label": pred_label,
        "correct": int(pred_label == TRUE_LABEL),
        "time_sec": end_time - start_time,
        "memory_MB": end_mem - start_mem
    })

# RESULT
df = pd.DataFrame(results)

df.to_csv("efficientnet_results.csv", index=False)

print(df)

print("\nAccuracy:", df["correct"].mean())
print("Avg Time:", df["time_sec"].mean())
print("Avg Memory:", df["memory_MB"].mean())

Using device: cpu
             video true_label pred_label  correct  time_sec   memory_MB
0  sign_379329.mp4      House     GO-OUT        0  0.560370  192.621094
1  sign_387006.mp4      House        CAR        0  0.642586   33.335938
2  sign_388681.mp4      House        CAR        0  0.610423    2.125000

Accuracy: 0.0
Avg Time: 0.6044596831003824
Avg Memory: 76.02734375


X3D

In [ ]:

# CONFIG
NUM_FRAMES = 16
IMG_SIZE = 224

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

video_folder = "test\CAR"
model_path = "x3d_s_13_classess_final.pth"

TRUE_LABEL = "House"

class_names = ['BUY','CAR','Eat','FRIEND','GO','GO-OUT',
               'HOUSE','Have','KISS-FIST','LOOK','Must',
               'NOW','TEND']

# MEDIAPIPE
MODEL_PATH = "hand_landmarker.task"

base_options = BaseOptions(model_asset_path=MODEL_PATH)

options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2
)

detector = vision.HandLandmarker.create_from_options(options)

# TRANSFORM
test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# MODEL (X3D-S)
model = x3d_s(pretrained=False)

model.blocks[5].proj = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.blocks[5].proj.in_features, len(class_names))
)

model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval()

# TESTING 
process = psutil.Process(os.getpid())
results = []

# handle single / folder
if os.path.isfile(video_folder):
    video_list = [video_folder]
else:
    video_list = [
        os.path.join(video_folder, v)
        for v in os.listdir(video_folder)
        if v.lower().endswith((".mp4", ".avi", ".mov"))
    ]

for video_path in video_list:

    video_name = os.path.basename(video_path)

    start_time = time.time()
    start_mem = process.memory_info().rss / (1024**2)

    video_tensor = video_to_tensor(video_path)
    video_tensor = video_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(video_tensor)
        pred_class = outputs.argmax(1).item()
        pred_label = class_names[pred_class]

    end_time = time.time()
    end_mem = process.memory_info().rss / (1024**2)

    results.append({
        "video": video_name,
        "true_label": TRUE_LABEL,
        "pred_label": pred_label,
        "correct": int(pred_label == TRUE_LABEL),
        "time_sec": end_time - start_time,
        "memory_MB": end_mem - start_mem
    })

# RESULTS
df = pd.DataFrame(results)

df.to_csv("x3d_results.csv", index=False)

print(df)

print("\nAccuracy:", df["correct"].mean())
print("Avg Time:", df["time_sec"].mean())
print("Avg Memory:", df["memory_MB"].mean())

Using device: cpu
             video true_label pred_label  correct  time_sec   memory_MB
0  sign_379329.mp4      House        CAR        0  0.448501  181.539062
1  sign_387006.mp4      House        CAR        0  0.601821   10.257812
2  sign_388681.mp4      House        CAR        0  0.574484    2.863281

Accuracy: 0.0
Avg Time: 0.5416018962860107
Avg Memory: 64.88671875


Static Image Test Setup

In [8]:
def video_to_frames(video_path, output_size=(224,224)):

    frames = load_frames_opencv(video_path)
    if len(frames) == 0:
        raise ValueError("No frames found in video")

    # 2. Get hand bounding box
    bbox = get_static_hand_bbox(frames)

    # 3. Crop frames
    cropped_frames = crop_frames_with_bbox(frames, bbox)

    # 4. Resize
    cropped_frames = [cv2.resize(frame, output_size) for frame in cropped_frames]

    return cropped_frames

ResNet

In [ ]:
process = psutil.Process(os.getpid())


# CONFIG
video_folder = "test/CAR"
model_path = "resnet18_0.4565_fixed.pth"   # <-- CHANGE THIS

TRUE_LABEL = "CAR"

class_names = ['BUY','CAR','Eat','FRIEND','GO','GO-OUT',
               'HOUSE','Have','KISS-FIST','LOOK','Must',
               'NOW','TEND']

device = "cuda" if torch.cuda.is_available() else "cpu"


# Augmentation
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# LOAD MODEL 
model = timm.create_model("resnet18", pretrained=False)  # <-- match training!

in_features = model.get_classifier().in_features

model.fc = nn.Sequential(
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.2),
    nn.Linear(in_features, len(class_names))
)

model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()


# INFERENCE LOOP
results = []

for video in os.listdir(video_folder):

    video_path = os.path.join(video_folder, video)

    # Timing
    start_time = time.time()
    start_mem = process.memory_info().rss / (1024**2)

    # Frames
    frames = video_to_frames(video_path)

    # Preprocess
    processed_frames = torch.stack([
        transform(frame) for frame in frames
    ]).to(device)

    # Inference
    with torch.no_grad():
        outputs = model(processed_frames)
        probs = torch.softmax(outputs, dim=1)
        all_frame_preds = probs.cpu().numpy()

    # Aggregate (mean over frames)
    video_confidence_vector = np.mean(all_frame_preds, axis=0)

    pred_idx = np.argmax(video_confidence_vector)
    pred_label = class_names[pred_idx]
    confidence = video_confidence_vector[pred_idx]

    # Timing end
    end_time = time.time()
    end_mem = process.memory_info().rss / (1024**2)

    # Store results
    results.append({
        "video": video,
        "true_label": TRUE_LABEL,
        "pred_label": pred_label,
        "confidence": float(confidence),
        "correct": int(pred_label == TRUE_LABEL),
        "time_sec": end_time - start_time,
        "memory_MB": end_mem - start_mem
    })

# SAVE RESULTS
df = pd.DataFrame(results)
df.to_csv("single_class_test.csv", index=False)

print(df)
print("\nAccuracy (this class only):", df["correct"].mean())
print("Avg Time:", df["time_sec"].mean())
print("Avg Memory:", df["memory_MB"].mean())

✅ Model loaded!
             video true_label pred_label  confidence  correct  time_sec  \
0  sign_379329.mp4        CAR        Eat    0.711295        0  0.608034   

   memory_MB  
0  66.171875  

Accuracy (this class only): 0.0
Avg Time: 0.6080338954925537
Avg Memory: 66.171875


MobileNet

In [ ]:
process = psutil.Process(os.getpid())

# CONFIG
video_folder = "test/CAR"
model_path = "mobilenet_0.4258_fixed.pth"   # <-- CHANGE THIS

TRUE_LABEL = "CAR"

class_names = ['BUY','CAR','Eat','FRIEND','GO','GO-OUT',
               'HOUSE','Have','KISS-FIST','LOOK','Must',
               'NOW','TEND']

device = "cuda" if torch.cuda.is_available() else "cpu"


# Augmentation
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# LOAD MODEL (MATCH TRAINING EXACTLY)
model = timm.create_model("mobilenetv2_100", pretrained=False)  # <-- match training!

in_features = model.get_classifier().in_features

model.classifier  = nn.Sequential(
    nn.BatchNorm1d(in_features),
    nn.Dropout(0.2),
    nn.Linear(in_features, len(class_names))
)

model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# INFERENCE LOOP
results = []

for video in os.listdir(video_folder):

    video_path = os.path.join(video_folder, video)

    # Timing
    start_time = time.time()
    start_mem = process.memory_info().rss / (1024**2)

    # Frames
    frames = video_to_frames(video_path)

    # Preprocess
    processed_frames = torch.stack([
        transform(frame) for frame in frames
    ]).to(device)

    # Inference
    with torch.no_grad():
        outputs = model(processed_frames)
        probs = torch.softmax(outputs, dim=1)
        all_frame_preds = probs.cpu().numpy()

    # Aggregate (mean over frames)
    video_confidence_vector = np.mean(all_frame_preds, axis=0)

    pred_idx = np.argmax(video_confidence_vector)
    pred_label = class_names[pred_idx]
    confidence = video_confidence_vector[pred_idx]

    # Timing end
    end_time = time.time()
    end_mem = process.memory_info().rss / (1024**2)

    # Store results
    results.append({
        "video": video,
        "true_label": TRUE_LABEL,
        "pred_label": pred_label,
        "confidence": float(confidence),
        "correct": int(pred_label == TRUE_LABEL),
        "time_sec": end_time - start_time,
        "memory_MB": end_mem - start_mem
    })

# SAVE RESULTS
df = pd.DataFrame(results)
df.to_csv("single_class_test.csv", index=False)

print(df)
print("\nAccuracy (this class only):", df["correct"].mean())
print("Avg Time:", df["time_sec"].mean())
print("Avg Memory:", df["memory_MB"].mean())

✅ Model loaded!
             video true_label pred_label  confidence  correct  time_sec  \
0  sign_379329.mp4        CAR        CAR    0.576975        1  0.662871   
1  sign_387006.mp4        CAR        CAR    0.597308        1  0.788405   
2  sign_388681.mp4        CAR        CAR    0.605302        1  0.775306   

    memory_MB  
0  241.921875  
1    5.644531  
2   -0.835938  

Accuracy (this class only): 1.0
Avg Time: 0.742193857828776
Avg Memory: 82.24348958333333
